# Fire PM2.5 raw-flux U-Net: fixed random 10-fold CV

This notebook reproduces the current champion setup:

- predictor: raw emission flux + lag-30 weather tensor
- target: raw Fire PM2.5
- U-Net dropout: all zero
- loss-only target cap: 200
- Huber beta: 25
- reporting high threshold: 5
- 2000–2020: a locked, fully random 10-fold split
- 2021–2023: external validation only

For every selected fold, the notebook saves the held-out test predictions and the
2021–2023 external predictions as compressed pixel arrays plus daily/yearly CSV files.
Raw targets used for metrics and export are not clipped by the loss cap.


In [ ]:
import argparse
import gc
import hashlib
import json
import math
import os
import random
import time
from dataclasses import asdict, dataclass
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)


In [ ]:
# A single integer trains one fold; a list trains multiple folds.
# Examples:
# FOLD = 0
# FOLD = [0, 1, 2]
# FOLD = list(range(10))
FOLD = list(range(10))

# Requested plotting exports.
SAVE_TEST_PIXEL_OUTPUTS = True
SAVE_EXTERNAL_PIXEL_OUTPUTS = True

# Resume controls. With AUTO_RESUME=True, an existing fold_xx/last.pt is resumed.
AUTO_RESUME = False
RESUME_PATH = None
ADDITIONAL_EPOCHS_WHEN_RESUMING = 100

DATA_ROOT = Path(os.environ.get("AU_FIRE_ROOT", str(Path.cwd().parent / "data")))

args = argparse.Namespace(
    input_dir=str(DATA_ROOT / "tensor/predictor_weather_lag30_totemi_raw"),
    output_dir=str(DATA_ROOT / "tensor/firepm25"),
    stats_path=str(DATA_ROOT / "tensor/predictor_weather_lag30_totemi_raw.pt"),
    manifest_path=str(DATA_ROOT / "splits/firepm25_rawflux_2000-2020_locked_random_10fold_champion_aligned_seed42.csv"),
    external_manifest_path=str(DATA_ROOT / "splits/firepm25_rawflux_external_2021-2023.csv"),
    run_dir=str(DATA_ROOT / "checkpoints/firepm25_rawflux_unet_cv10_champion_aligned_cap200_beta25_high5"),

    input_channels=45,
    output_channels=1,
    base_channels=64,
    encoder_dropout=0.0,
    bottleneck_dropout=0.0,
    decoder_dropout_1=0.0,
    decoder_dropout_2=0.0,
    decoder_dropout_3=0.0,
    decoder_dropout_4=0.0,
    use_dropout2d=True,
    dropout_position="between",
    use_batchnorm=True,
    batchnorm_momentum=0.1,
    activation="relu",

    num_epochs=250,
    batch_size=64,
    learning_rate=1e-2,
    weight_decay=1e-5,
    gradient_clip_norm=5.0,
    # Exact settings from the run that produced the random42_train90 result.
    lr_plateau_patience=4,
    scheduler_gamma=0.5,
    scheduler_min_lr=1e-4,
    early_stopping_patience=8,
    # Parallel network-file loading with bounded prefetch and worker reuse.
    num_workers=20,
    prefetch_factor=1,
    persistent_workers=True,
    load_retries=3,
    load_retry_delay=0.15,

    loss_name="raw_huber",
    y_transform="raw",
    huber_beta=25.0,
    loss_max_threshold=200.0,
    raw_prediction_cap=1000.0,
    high_threshold=5,
    max_threshold=None,

    experiment_seed=42,
    split_seed=42,
    device_ids=[0],
)

if torch.cuda.is_available() and args.device_ids:
    device = torch.device(f"cuda:{args.device_ids[0]}")
    torch.cuda.set_device(device)
else:
    device = torch.device("cpu")

# Avoid the file-descriptor sharing strategy accumulating many open handles.
if args.num_workers > 0 and "file_system" in torch.multiprocessing.get_all_sharing_strategies():
    torch.multiprocessing.set_sharing_strategy("file_system")

Path(args.run_dir).mkdir(parents=True, exist_ok=True)
print(f"Device: {device}")
print(f"Run directory: {args.run_dir}")
print(
    f"Champion loss: raw Huber | beta={args.huber_beta:g} | "
    f"loss cap={args.loss_max_threshold:g} | high threshold={args.high_threshold:g}"
)


In [ ]:
def normalize_selected_folds(value):
    if isinstance(value, (int, np.integer)):
        folds = [int(value)]
    elif isinstance(value, (list, tuple, set, np.ndarray)):
        folds = [int(v) for v in value]
    else:
        raise TypeError("FOLD must be an integer or a sequence of integers")
    if not folds or len(set(folds)) != len(folds):
        raise ValueError("FOLD must contain unique fold IDs")
    if any(f < 0 or f > 9 for f in folds):
        raise ValueError("Every fold ID must be from 0 to 9")
    return folds


def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


selected_folds = normalize_selected_folds(FOLD)
fold_manifest = pd.read_csv(args.manifest_path)
external_manifest = pd.read_csv(args.external_manifest_path)

required_fold_columns = {"filename", "date", "fold", "random_position"}
if not required_fold_columns.issubset(fold_manifest.columns):
    raise ValueError(f"Fold manifest requires columns: {sorted(required_fold_columns)}")
if set(fold_manifest["fold"].astype(int).unique()) != set(range(10)):
    raise ValueError("The locked manifest must contain fold IDs 0 through 9")
if fold_manifest["filename"].duplicated().any():
    raise ValueError("Duplicate development filenames found")
if external_manifest["filename"].duplicated().any():
    raise ValueError("Duplicate external filenames found")
if set(fold_manifest["filename"]) & set(external_manifest["filename"]):
    raise ValueError("Development and external manifests overlap")

fold_manifest["fold"] = fold_manifest["fold"].astype(int)
fold_manifest["date"] = pd.to_datetime(fold_manifest["date"])
external_manifest["date"] = pd.to_datetime(external_manifest["date"])
manifest_hash = sha256_file(args.manifest_path)
external_manifest_hash = sha256_file(args.external_manifest_path)

print("Selected folds:")
for fold_id in selected_folds:
    print(f"  - {fold_id}")
print("Locked fold sizes:")
print(fold_manifest.groupby("fold").size().rename("n_days").to_frame())
print(f"Development days: {len(fold_manifest):,}")
print(f"External days: {len(external_manifest):,}")
print(f"Fold manifest SHA256: {manifest_hash}")


In [ ]:
def safe_tensor_load(path):
    path = Path(path)
    for attempt in range(args.load_retries):
        try:
            return torch.load(path, map_location="cpu")
        except OSError:
            if attempt + 1 >= args.load_retries:
                raise
            time.sleep(args.load_retry_delay * (attempt + 1))


climate_mean, climate_std = safe_tensor_load(args.stats_path)
climate_mean = torch.as_tensor(climate_mean, dtype=torch.float32).flatten()
climate_std = torch.as_tensor(climate_std, dtype=torch.float32).flatten()
if climate_mean.numel() != 41 or climate_std.numel() != 41:
    raise ValueError(
        f"Expected 41 predictor statistics, got mean={climate_mean.numel()}, std={climate_std.numel()}"
    )
print(f"Loaded normalization statistics for {climate_mean.numel()} predictor channels")

class AirPollutionDataset(Dataset):
    def __init__(self, filenames, x_dir, y_dir, max_threshold=None, mean=None, std=None):
        self.filenames = list(filenames)
        self.x_dir = Path(x_dir)
        self.y_dir = Path(y_dir)
        self.max_threshold = max_threshold
        self.mean = mean
        self.std = std

        # Fixed spatial encoding for the 140 x 284 Australia crop.
        # Approximate native coordinate range:
        # latitude  -9.125 (north) to -43.875 (south)
        # longitude 96.875 (west) to 167.625 (east)
        # Both are linearly normalized to [-1, 1].
        height, width = 140, 284
        lat_norm = torch.linspace(1.0, -1.0, height, dtype=torch.float32)
        lon_norm = torch.linspace(-1.0, 1.0, width, dtype=torch.float32)
        lat_channel = lat_norm[:, None].expand(height, width).unsqueeze(0)
        lon_channel = lon_norm[None, :].expand(height, width).unsqueeze(0)
        self.spatial_encoding = torch.cat([lat_channel, lon_channel], dim=0)

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        fname = self.filenames[idx]
        x = safe_tensor_load(self.x_dir / fname).float()
        y = safe_tensor_load(self.y_dir / fname).float()
        mask = torch.isfinite(y)
        y = torch.nan_to_num(y, nan=0.0, posinf=0.0, neginf=0.0).clamp_min(0.0)

        if self.mean is not None and self.std is not None:
            x = (x - self.mean[:, None, None]) / (self.std[:, None, None] + 1e-6)
        x = torch.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)

        dt = datetime.strptime(fname.replace(".pt", ""), "%Y-%m-%d")
        doy = dt.timetuple().tm_yday
        period = 365.2425
        sin_doy = torch.full((1, *x.shape[-2:]), math.sin(2 * math.pi * doy / period))
        cos_doy = torch.full((1, *x.shape[-2:]), math.cos(2 * math.pi * doy / period))

        if x.shape[-2:] != self.spatial_encoding.shape[-2:]:
            raise ValueError(f"Spatial size mismatch for {fname}: {x.shape[-2:]}")
        # Channel order: 41 predictors, sin/cos DOY, normalized latitude/longitude.
        x_full = torch.cat([x, sin_doy, cos_doy, self.spatial_encoding], dim=0)
        if x_full.shape[0] != args.input_channels:
            raise ValueError(f"Expected {args.input_channels} channels, got {x_full.shape[0]} for {fname}")

        if self.max_threshold is not None:
            y = torch.clamp(y, max=self.max_threshold)
        # Year is returned for optional per-batch year-macro loss.
        return x_full, y, mask.float(), torch.tensor(dt.year, dtype=torch.long)



def make_dataset(filenames):
    return AirPollutionDataset(
        filenames=filenames,
        x_dir=args.input_dir,
        y_dir=args.output_dir,
        max_threshold=args.max_threshold,
        mean=climate_mean,
        std=climate_std,
    )


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass, asdict
# =========================
# UNet 配置转换函数
# =========================
@dataclass
class UNetConfig:
    """UNet 模型配置"""
    in_channels: int = 50
    out_channels: int = 1
    base_channels: int = 64
    
    # Dropout 配置
    encoder_dropout: float = 0.0
    bottleneck_dropout: float = 0.4
    decoder_dropouts: tuple = (0.3, 0.2, 0.1, 0.0)
    
    # Dropout 选项
    use_dropout2d: bool = True
    dropout_position: str = 'between'
    
    # BatchNorm 配置
    use_batchnorm: bool = True
    batchnorm_momentum: float = 0.1
    
    # 激活函数
    activation: str = 'relu'
    
    def __post_init__(self):
        """验证配置参数"""
        assert len(self.decoder_dropouts) == 4, "decoder_dropouts 必须包含 4 个值"
        assert 0 <= self.encoder_dropout <= 1, "dropout 值必须在 [0, 1] 之间"
        assert 0 <= self.bottleneck_dropout <= 1, "dropout 值必须在 [0, 1] 之间"
        assert all(0 <= d <= 1 for d in self.decoder_dropouts), "dropout 值必须在 [0, 1] 之间"


def args_to_unet_config(args):
    """将 args 转换为 UNetConfig"""
    return UNetConfig(
        in_channels=args.input_channels,
        out_channels=args.output_channels,
        base_channels=args.base_channels,
        encoder_dropout=args.encoder_dropout,
        bottleneck_dropout=args.bottleneck_dropout,
        decoder_dropouts=(
            args.decoder_dropout_1,
            args.decoder_dropout_2,
            args.decoder_dropout_3,
            args.decoder_dropout_4
        ),
        use_dropout2d=args.use_dropout2d,
        dropout_position=args.dropout_position,
        use_batchnorm=args.use_batchnorm,
        batchnorm_momentum=args.batchnorm_momentum,
        activation=args.activation
    )


# =========================
# 基础模块
# =========================
class DoubleConv(nn.Module):
    """双卷积模块"""
    
    def __init__(self, in_channels, out_channels, config: UNetConfig, dropout=0.0):
        super().__init__()
        
        self.dropout = dropout
        self.config = config
        
        # 第一个卷积块
        conv1_layers = [
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False)
        ]
        if config.use_batchnorm:
            conv1_layers.append(nn.BatchNorm2d(out_channels, momentum=config.batchnorm_momentum))
        conv1_layers.append(self._get_activation())
        self.conv1 = nn.Sequential(*conv1_layers)
        
        # Dropout（在两个 conv 之间）
        if dropout > 0 and config.dropout_position == 'between':
            if config.use_dropout2d:
                self.dropout_layer = nn.Dropout2d(p=dropout)
            else:
                self.dropout_layer = nn.Dropout(p=dropout)
        else:
            self.dropout_layer = None
        
        # 第二个卷积块
        conv2_layers = [
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False)
        ]
        if config.use_batchnorm:
            conv2_layers.append(nn.BatchNorm2d(out_channels, momentum=config.batchnorm_momentum))
        conv2_layers.append(self._get_activation())
        self.conv2 = nn.Sequential(*conv2_layers)
        
        # Dropout（在两个 conv 之后）
        if dropout > 0 and config.dropout_position == 'after':
            if config.use_dropout2d:
                self.dropout_after = nn.Dropout2d(p=dropout)
            else:
                self.dropout_after = nn.Dropout(p=dropout)
        else:
            self.dropout_after = None

    def _get_activation(self):
        """根据配置返回激活函数"""
        if self.config.activation == 'relu':
            return nn.ReLU(inplace=True)
        elif self.config.activation == 'leaky_relu':
            return nn.LeakyReLU(0.2, inplace=True)
        elif self.config.activation == 'gelu':
            return nn.GELU()
        else:
            raise ValueError(f"不支持的激活函数: {self.config.activation}")

    def forward(self, x):
        x = self.conv1(x)
        if self.dropout_layer is not None:
            x = self.dropout_layer(x)
        x = self.conv2(x)
        if self.dropout_after is not None:
            x = self.dropout_after(x)
        return x


def pad_to_match(x, ref):
    """将 x 填充到与 ref 相同的尺寸"""
    diff_y = ref.size(2) - x.size(2)
    diff_x = ref.size(3) - x.size(3)
    return F.pad(x, [0, diff_x, 0, diff_y])


# =========================
# UNet 模型
# =========================
class UNet(nn.Module):
    """可配置的 UNet 模型"""
    
    def __init__(self, config: UNetConfig):
        super().__init__()
        self.config = config
        
        bc = config.base_channels
        
        # ==================== Encoder ====================
        self.enc1 = DoubleConv(config.in_channels, bc, config, dropout=config.encoder_dropout)
        self.pool1 = nn.MaxPool2d(2)
        
        self.enc2 = DoubleConv(bc, bc * 2, config, dropout=config.encoder_dropout)
        self.pool2 = nn.MaxPool2d(2)
        
        self.enc3 = DoubleConv(bc * 2, bc * 4, config, dropout=config.encoder_dropout)
        self.pool3 = nn.MaxPool2d(2)
        
        self.enc4 = DoubleConv(bc * 4, bc * 8, config, dropout=config.encoder_dropout)
        self.pool4 = nn.MaxPool2d(2)
        
        # ==================== Bottleneck ====================
        self.bottleneck = DoubleConv(
            bc * 8, 
            bc * 16, 
            config, 
            dropout=config.bottleneck_dropout
        )
        
        # ==================== Decoder ====================
        self.up4 = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True)
        self.dec4 = DoubleConv(bc * 16 + bc * 8, bc * 8, config, dropout=config.decoder_dropouts[0])
        
        self.up3 = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True)
        self.dec3 = DoubleConv(bc * 8 + bc * 4, bc * 4, config, dropout=config.decoder_dropouts[1])
        
        self.up2 = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True)
        self.dec2 = DoubleConv(bc * 4 + bc * 2, bc * 2, config, dropout=config.decoder_dropouts[2])
        
        self.up1 = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True)
        self.dec1 = DoubleConv(bc * 2 + bc, bc, config, dropout=config.decoder_dropouts[3])
        
        # ==================== Output ====================
        self.out_conv = nn.Conv2d(bc, config.out_channels, kernel_size=1)
        
        # 初始化权重
        self._initialize_weights()
    
    def _initialize_weights(self):
        """权重初始化"""
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
    
    def forward(self, x):
        # Encoder
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        e4 = self.enc4(self.pool3(e3))
        
        # Bottleneck
        b = self.bottleneck(self.pool4(e4))
        
        # Decoder
        u4 = self.up4(b)
        u4 = pad_to_match(u4, e4)
        d4 = self.dec4(torch.cat([u4, e4], dim=1))
        
        u3 = self.up3(d4)
        u3 = pad_to_match(u3, e3)
        d3 = self.dec3(torch.cat([u3, e3], dim=1))
        
        u2 = self.up2(d3)
        u2 = pad_to_match(u2, e2)
        d2 = self.dec2(torch.cat([u2, e2], dim=1))
        
        u1 = self.up1(d2)
        u1 = pad_to_match(u1, e1)
        d1 = self.dec1(torch.cat([u1, e1], dim=1))
        
        return self.out_conv(d1)


# =========================
# 模型初始化函数
# =========================
def create_model(args, verbose=True):
    """
    从 args 创建 UNet 模型
    
    Args:
        args: 包含所有超参数的 argparse.Namespace
        verbose: 是否打印模型信息
    
    Returns:
        model: UNet 模型
        config: UNetConfig 配置对象
    """
    # 转换配置
    config = args_to_unet_config(args)
    
    # 创建模型
    model = UNet(config)
    
    if verbose:
        print("=" * 70)
        print("🔥 UNet Model Configuration")
        print("=" * 70)
        print(f"📊 Architecture:")
        print(f"  - Input channels:     {config.in_channels}")
        print(f"  - Output channels:    {config.out_channels}")
        print(f"  - Base channels:      {config.base_channels}")
        print(f"\n🛡️  Dropout Configuration:")
        print(f"  - Encoder dropout:    {config.encoder_dropout}")
        print(f"  - Bottleneck dropout: {config.bottleneck_dropout}")
        print(f"  - Decoder dropouts:   {config.decoder_dropouts}")
        print(f"  - Dropout type:       {'Dropout2d' if config.use_dropout2d else 'Dropout'}")
        print(f"  - Dropout position:   {config.dropout_position}")
        print(f"\n⚙️  Other Settings:")
        print(f"  - BatchNorm:          {config.use_batchnorm}")
        print(f"  - Activation:         {config.activation}")
        print(f"  - Weight decay:       {args.weight_decay}")
        
        # 参数统计
        total_params = sum(p.numel() for p in model.parameters())
        trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"\n📈 Parameters:")
        print(f"  - Total:              {total_params:,}")
        print(f"  - Trainable:          {trainable_params:,}")
        print("=" * 70)
    
    # 多 GPU 设置
    if len(args.device_ids) > 1:
        model = nn.DataParallel(model, device_ids=args.device_ids)
        if verbose:
            print(f"✅ Using DataParallel with GPUs: {args.device_ids}")
    
    # 移动到设备
    device = torch.device(f"cuda:{args.device_ids[0]}" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    
    return model, config


In [ ]:
class RawHuberLoss(nn.Module):
    def __init__(self, beta, loss_max_threshold, raw_prediction_cap):
        super().__init__()
        self.beta = float(beta)
        self.loss_max_threshold = (
            None if loss_max_threshold is None else float(loss_max_threshold)
        )
        self.raw_prediction_cap = float(raw_prediction_cap)

    @staticmethod
    def macro_day_mean(pixel_loss, mask):
        dims = tuple(range(1, pixel_loss.ndim))
        denominator = mask.sum(dim=dims)
        valid_days = denominator > 0
        per_day = (pixel_loss * mask).sum(dim=dims) / denominator.clamp_min(1.0)
        if not torch.any(valid_days):
            return pixel_loss.sum() * 0.0
        return per_day[valid_days].mean()

    def forward(self, model_output, target, mask):
        target = target.float().clamp_min(0.0)
        mask = (mask > 0).float()
        if target.ndim == model_output.ndim - 1:
            target = target.unsqueeze(1)
            mask = mask.unsqueeze(1)
        prediction_for_loss = F.softplus(model_output.float())
        loss_target = target
        if self.loss_max_threshold is not None:
            loss_target = loss_target.clamp(max=self.loss_max_threshold)
        pixel_loss = F.smooth_l1_loss(
            prediction_for_loss,
            loss_target,
            beta=self.beta,
            reduction="none",
        )
        # Match the original: the safety cap affects raw metrics/export, not Huber loss.
        prediction_raw = prediction_for_loss.clamp(max=self.raw_prediction_cap)
        return self.macro_day_mean(pixel_loss, mask), prediction_raw


criterion = RawHuberLoss(
    beta=args.huber_beta,
    loss_max_threshold=args.loss_max_threshold,
    raw_prediction_cap=args.raw_prediction_cap,
)


class MetricAccumulator:
    def __init__(self, high_threshold):
        self.high_threshold = float(high_threshold)
        self.loss_weighted_sum = 0.0
        self.loss_days = 0
        self.n = 0.0
        self.sse = 0.0
        self.sae = 0.0
        self.sum_error = 0.0
        self.sum_y = 0.0
        self.sum_y2 = 0.0
        self.low_sse = 0.0
        self.low_n = 0.0
        self.high_sse = 0.0
        self.high_n = 0.0

    def update(self, loss, prediction, target, mask):
        target = target.float()
        mask = (mask > 0)
        if target.ndim == prediction.ndim - 1:
            target = target.unsqueeze(1)
            mask = mask.unsqueeze(1)
        valid = mask
        n = valid.sum().item()
        if n <= 0:
            return
        error = (prediction - target)
        error_valid = error[valid]
        target_valid = target[valid]
        self.loss_weighted_sum += float(loss.detach()) * target.shape[0]
        self.loss_days += target.shape[0]
        self.n += n
        self.sse += error_valid.square().double().sum().item()
        self.sae += error_valid.abs().double().sum().item()
        self.sum_error += error_valid.double().sum().item()
        self.sum_y += target_valid.double().sum().item()
        self.sum_y2 += target_valid.double().square().sum().item()
        low = valid & (target <= self.high_threshold)
        high = valid & (target > self.high_threshold)
        self.low_n += low.sum().item()
        self.high_n += high.sum().item()
        self.low_sse += error[low].double().square().sum().item()
        self.high_sse += error[high].double().square().sum().item()

    def compute(self):
        n = max(self.n, 1.0)
        sst = self.sum_y2 - self.sum_y * self.sum_y / n
        return {
            "loss": self.loss_weighted_sum / max(self.loss_days, 1),
            "rmse": math.sqrt(max(self.sse, 0.0) / n),
            "mae": self.sae / n,
            "bias": self.sum_error / n,
            "r2": 0.0 if sst <= 1e-12 else 1.0 - self.sse / sst,
            "low_mse": self.low_sse / max(self.low_n, 1.0),
            "high_mse": self.high_sse / max(self.high_n, 1.0),
            "n_pixels": int(self.n),
        }


def metrics_from_numpy(y_true, y_pred, valid_mask, high_threshold):
    target = torch.from_numpy(y_true)
    prediction = torch.from_numpy(y_pred)
    mask = torch.from_numpy(valid_mask.astype(bool))
    dummy = torch.tensor(0.0)
    acc = MetricAccumulator(high_threshold)
    acc.update(dummy, prediction, target, mask)
    result = acc.compute()
    result.pop("loss", None)
    return result


In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_loader(dataset, shuffle, seed):
    generator = torch.Generator()
    generator.manual_seed(seed)
    loader_kwargs = dict(
        dataset=dataset,
        batch_size=args.batch_size,
        shuffle=shuffle,
        num_workers=args.num_workers,
        pin_memory=(device.type == "cuda"),
        generator=generator,
    )
    if args.num_workers > 0:
        loader_kwargs["prefetch_factor"] = args.prefetch_factor
        # Retain only training workers; evaluation/export workers close after use.
        loader_kwargs["persistent_workers"] = bool(args.persistent_workers and shuffle)
    return DataLoader(**loader_kwargs)


def get_plain_model(model):
    return model.module if isinstance(model, nn.DataParallel) else model


def clean_state_dict(state_dict):
    return {
        (key[7:] if key.startswith("module.") else key): value
        for key, value in state_dict.items()
    }


def initialize_weights_like_original(module):
    if isinstance(module, nn.Conv2d):
        nn.init.kaiming_normal_(module.weight, mode="fan_out", nonlinearity="relu")
        if module.bias is not None:
            nn.init.zeros_(module.bias)
    elif isinstance(module, nn.BatchNorm2d):
        nn.init.ones_(module.weight)
        nn.init.zeros_(module.bias)


def build_model():
    model = UNet(args_to_unet_config(args))
    # The original notebook applies this initialization after UNet construction.
    model.apply(initialize_weights_like_original)
    model = model.to(device)
    if device.type == "cuda" and len(args.device_ids) > 1:
        model = nn.DataParallel(model, device_ids=args.device_ids)
    return model


def train_one_epoch(model, loader, optimizer, scaler):
    model.train()
    acc = MetricAccumulator(args.high_threshold)
    progress = tqdm(loader, desc="training", leave=False)
    for x, y, mask, _ in progress:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        mask = mask.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(
            device_type=device.type,
            enabled=(device.type == "cuda"),
        ):
            output = model(x)
            loss, prediction = criterion(output, y, mask)
        if not torch.isfinite(loss):
            raise FloatingPointError(f"Non-finite training loss: {float(loss)}")
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=args.gradient_clip_norm,
        )
        scaler.step(optimizer)
        scaler.update()
        acc.update(loss, prediction.detach(), y, mask)
        progress.set_postfix(loss=f"{float(loss):.4f}")
    return acc.compute()


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    acc = MetricAccumulator(args.high_threshold)
    for x, y, mask, _ in tqdm(loader, desc="evaluating", leave=False):
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        mask = mask.to(device, non_blocking=True)
        with torch.amp.autocast(
            device_type=device.type,
            enabled=(device.type == "cuda"),
        ):
            output = model(x)
            loss, prediction = criterion(output, y, mask)
        acc.update(loss, prediction, y, mask)
    return acc.compute()


def format_metrics(label, stats):
    return (
        f"  {label:<12} loss={stats['loss']:.5f} | RMSE={stats['rmse']:.4f} | "
        f"MAE={stats['mae']:.4f} | Bias={stats['bias']:.4f} | R2={stats['r2']:.4f} | "
        f"LowMSE={stats['low_mse']:.4f} | HighMSE={stats['high_mse']:.4f}"
    )


def checkpoint_payload(
    model,
    optimizer,
    scheduler,
    scaler,
    fold_id,
    epoch,
    best_r2,
    best_epoch,
    no_improve,
):
    return {
        "model": get_plain_model(model).state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "scheduler_type": "ReduceLROnPlateau",
        "scaler": scaler.state_dict(),
        "fold": int(fold_id),
        "epoch": int(epoch),
        "best_r2": float(best_r2),
        "best_epoch": int(best_epoch),
        "no_improve": int(no_improve),
        "config": vars(args),
        "manifest_sha256": manifest_hash,
        "external_manifest_sha256": external_manifest_hash,
    }


def validate_checkpoint(checkpoint, fold_id):
    if int(checkpoint.get("fold", -1)) != int(fold_id):
        raise ValueError("Checkpoint fold does not match the selected fold")
    if checkpoint.get("manifest_sha256") != manifest_hash:
        raise ValueError("Checkpoint was trained with a different fold manifest")
    saved = checkpoint.get("config", {})
    keys = [
        "input_channels",
        "base_channels",
        "huber_beta",
        "loss_max_threshold",
        "high_threshold",
        "encoder_dropout",
        "bottleneck_dropout",
        "decoder_dropout_1",
        "decoder_dropout_2",
        "decoder_dropout_3",
        "decoder_dropout_4",
    ]
    mismatches = {
        key: (saved.get(key), getattr(args, key))
        for key in keys
        if saved.get(key) != getattr(args, key)
    }
    if mismatches:
        raise ValueError(f"Checkpoint configuration mismatch: {mismatches}")


In [ ]:
@torch.no_grad()
def export_predictions(model, dataset, split_name, fold_id, output_dir, save_pixels):
    loader = make_loader(dataset, shuffle=False, seed=args.experiment_seed + fold_id)
    y_true_batches = []
    y_pred_batches = []
    mask_batches = []
    daily_rows = []
    offset = 0
    model.eval()

    for x, y, mask, _ in tqdm(loader, desc=f"exporting {split_name}", leave=False):
        x = x.to(device, non_blocking=True)
        with torch.amp.autocast(
            device_type=device.type,
            enabled=(device.type == "cuda"),
        ):
            output = model(x)
            prediction = F.softplus(output.float()).clamp(max=args.raw_prediction_cap)

        y_np = y.numpy().astype(np.float32, copy=False)
        pred_np = prediction.squeeze(1).cpu().numpy().astype(np.float32, copy=False)
        mask_np = (mask.numpy() > 0)
        if y_np.ndim == 4 and y_np.shape[1] == 1:
            y_np = y_np[:, 0]
        if mask_np.ndim == 4 and mask_np.shape[1] == 1:
            mask_np = mask_np[:, 0]

        for batch_index in range(y_np.shape[0]):
            valid = mask_np[batch_index]
            date = Path(dataset.filenames[offset + batch_index]).stem
            metrics = metrics_from_numpy(
                y_np[batch_index],
                pred_np[batch_index],
                valid,
                args.high_threshold,
            )
            daily_rows.append({"date": date, "fold": fold_id, "split": split_name, **metrics})
        offset += y_np.shape[0]

        y_true_batches.append(y_np)
        y_pred_batches.append(pred_np)
        mask_batches.append(mask_np)

    y_true = np.concatenate(y_true_batches, axis=0)
    y_pred = np.concatenate(y_pred_batches, axis=0)
    valid_mask = np.concatenate(mask_batches, axis=0)
    dates = np.asarray([Path(name).stem for name in dataset.filenames])
    overall = metrics_from_numpy(y_true, y_pred, valid_mask, args.high_threshold)
    overall.update({"fold": fold_id, "split": split_name, "n_days": len(dates)})

    daily = pd.DataFrame(daily_rows)
    daily["date"] = pd.to_datetime(daily["date"])
    daily["year"] = daily["date"].dt.year
    yearly_rows = []
    for year, indices in daily.groupby("year").groups.items():
        idx = np.asarray(list(indices), dtype=int)
        yearly = metrics_from_numpy(
            y_true[idx],
            y_pred[idx],
            valid_mask[idx],
            args.high_threshold,
        )
        yearly_rows.append(
            {
                "fold": fold_id,
                "split": split_name,
                "year": int(year),
                "n_days": len(idx),
                **yearly,
            }
        )
    yearly = pd.DataFrame(yearly_rows)

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    daily.to_csv(output_dir / f"{split_name}_daily_metrics.csv", index=False)
    yearly.to_csv(output_dir / f"{split_name}_yearly_metrics.csv", index=False)
    pd.DataFrame([overall]).to_csv(
        output_dir / f"{split_name}_overall_metrics.csv",
        index=False,
    )

    if save_pixels:
        np.savez_compressed(
            output_dir / f"{split_name}_pixel_predictions.npz",
            y_true=y_true,
            y_pred=y_pred,
            valid_mask=valid_mask.astype(np.uint8),
            dates=dates,
            fold=np.asarray([fold_id], dtype=np.int16),
            split=np.asarray([split_name]),
            high_threshold=np.asarray([args.high_threshold], dtype=np.float32),
            loss_max_threshold=np.asarray([args.loss_max_threshold], dtype=np.float32),
            huber_beta=np.asarray([args.huber_beta], dtype=np.float32),
        )

    return overall, daily, yearly


def resolve_resume_path(fold_dir, fold_id):
    if RESUME_PATH is not None:
        if len(selected_folds) != 1:
            raise ValueError("Explicit RESUME_PATH can only be used with one selected fold")
        return Path(RESUME_PATH)
    automatic = fold_dir / "last.pt"
    return automatic if AUTO_RESUME and automatic.is_file() else None


In [ ]:
def run_fold(fold_id):
    set_seed(args.experiment_seed)
    fold_dir = Path(args.run_dir) / f"fold_{fold_id:02d}"
    fold_dir.mkdir(parents=True, exist_ok=True)

    # Preserve the exact torch random_split permutation order. For fold 0 this
    # reproduces both the champion training list and champion tune list.
    ordered_manifest = fold_manifest.sort_values("random_position")
    train_filenames = ordered_manifest.loc[
        ordered_manifest["fold"] != fold_id,
        "filename",
    ].tolist()
    test_filenames = ordered_manifest.loc[
        ordered_manifest["fold"] == fold_id,
        "filename",
    ].tolist()
    external_filenames = external_manifest["filename"].tolist()

    train_dataset = make_dataset(train_filenames)
    test_dataset = make_dataset(test_filenames)
    external_dataset = make_dataset(external_filenames)
    train_loader = make_loader(train_dataset, shuffle=True, seed=args.experiment_seed)
    test_loader = make_loader(test_dataset, shuffle=False, seed=args.experiment_seed)

    pd.DataFrame({"filename": train_filenames}).to_csv(
        fold_dir / "train_filenames.csv",
        index=False,
    )
    pd.DataFrame({"filename": test_filenames}).to_csv(
        fold_dir / "test_filenames.csv",
        index=False,
    )

    model = build_model()
    optimizer = AdamW(
        model.parameters(),
        lr=args.learning_rate,
        weight_decay=args.weight_decay,
    )
    scheduler = ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=args.scheduler_gamma,
        patience=args.lr_plateau_patience,
        min_lr=args.scheduler_min_lr,
    )
    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=(device.type == "cuda"),
    )

    start_epoch = 0
    best_r2 = -float("inf")
    best_epoch = -1
    no_improve = 0
    resume_path = resolve_resume_path(fold_dir, fold_id)
    if resume_path is not None:
        checkpoint = torch.load(resume_path, map_location=device)
        validate_checkpoint(checkpoint, fold_id)
        get_plain_model(model).load_state_dict(clean_state_dict(checkpoint["model"]))
        optimizer.load_state_dict(checkpoint["optimizer"])
        if checkpoint.get("scheduler_type") == "ReduceLROnPlateau":
            scheduler.load_state_dict(checkpoint["scheduler"])
        else:
            print("  Old StepLR checkpoint detected: starting a fresh R2-plateau counter.")
        if "scaler" in checkpoint:
            scaler.load_state_dict(checkpoint["scaler"])
        start_epoch = int(checkpoint["epoch"]) + 1
        best_r2 = float(checkpoint["best_r2"])
        best_epoch = int(checkpoint["best_epoch"])
        no_improve = int(checkpoint.get("no_improve", 0))
        training_end_epoch = start_epoch + ADDITIONAL_EPOCHS_WHEN_RESUMING
        print(f"Fold {fold_id:02d}: resuming {resume_path} from epoch {start_epoch + 1}")
    else:
        training_end_epoch = args.num_epochs

    print("=" * 100)
    print(
        f"FOLD {fold_id:02d} | train={len(train_dataset):,} days | "
        f"test={len(test_dataset):,} days | external={len(external_dataset):,} days"
    )
    print("=" * 100)

    history = []
    best_path = fold_dir / "best.pt"
    last_path = fold_dir / "last.pt"
    for epoch in range(start_epoch, training_end_epoch):
        train_stats = train_one_epoch(model, train_loader, optimizer, scaler)
        test_stats = evaluate(model, test_loader)
        lr_used = optimizer.param_groups[0]["lr"]
        # Same checkpoint rule as the original notebook.
        improved = test_stats["r2"] > best_r2 + 1e-4
        if improved:
            best_r2 = test_stats["r2"]
            best_epoch = epoch
            no_improve = 0
        else:
            no_improve += 1

        scheduler.step(test_stats["r2"])
        next_lr = optimizer.param_groups[0]["lr"]
        lr_reduced = next_lr < lr_used

        row = {
            "fold": fold_id,
            "epoch": epoch + 1,
            "learning_rate_used": lr_used,
            "next_learning_rate": next_lr,
            "lr_reduced": lr_reduced,
            "is_best": improved,
            **{f"train_{k}": v for k, v in train_stats.items()},
            **{f"test_{k}": v for k, v in test_stats.items()},
        }
        history.append(row)
        pd.DataFrame(history).to_csv(fold_dir / "training_history.csv", index=False)

        payload = checkpoint_payload(
            model,
            optimizer,
            scheduler,
            scaler,
            fold_id,
            epoch,
            best_r2,
            best_epoch,
            no_improve,
        )
        torch.save(payload, last_path)
        if improved:
            torch.save(payload, best_path)

        print("-" * 100)
        print(
            f"Fold {fold_id:02d} | Epoch {epoch + 1:03d}/{training_end_epoch:03d} | "
            f"LR={lr_used:.3e}"
            + (f" -> {next_lr:.3e} [REDUCED]" if lr_reduced else "")
            + f" | Best={'YES' if improved else 'no'} | Best epoch={best_epoch + 1:03d}"
        )
        print(format_metrics("Train", train_stats))
        print(format_metrics("Test", test_stats))

        if no_improve >= args.early_stopping_patience:
            print(
                f"  Early stopping: {no_improve} epochs without a better held-out test R2."
            )
            break

    if not best_path.is_file():
        raise FileNotFoundError(f"Best checkpoint was not created: {best_path}")
    best_checkpoint = torch.load(best_path, map_location=device)
    validate_checkpoint(best_checkpoint, fold_id)
    get_plain_model(model).load_state_dict(clean_state_dict(best_checkpoint["model"]))

    # Shut down persistent training workers before starting export workers.
    del train_loader, test_loader
    gc.collect()

    test_overall, test_daily, test_yearly = export_predictions(
        model,
        test_dataset,
        "test_2000_2020",
        fold_id,
        fold_dir,
        SAVE_TEST_PIXEL_OUTPUTS,
    )
    external_overall, external_daily, external_yearly = export_predictions(
        model,
        external_dataset,
        "external_2021_2023",
        fold_id,
        fold_dir,
        SAVE_EXTERNAL_PIXEL_OUTPUTS,
    )

    print("=" * 100)
    print(f"FOLD {fold_id:02d} FINAL | best epoch={int(best_checkpoint['best_epoch']) + 1:03d}")
    print(format_metrics("Test", {"loss": float("nan"), **test_overall}))
    print(format_metrics("External", {"loss": float("nan"), **external_overall}))
    print(f"Saved outputs: {fold_dir}")

    summary = {
        "fold": fold_id,
        "best_epoch": int(best_checkpoint["best_epoch"]) + 1,
        **{f"test_{k}": v for k, v in test_overall.items() if k not in {"fold", "split"}},
        **{
            f"external_{k}": v
            for k, v in external_overall.items()
            if k not in {"fold", "split"}
        },
    }
    del model, optimizer, scheduler, scaler
    del train_dataset, test_dataset, external_dataset
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return summary, test_daily, test_yearly, external_daily, external_yearly


In [ ]:
fold_summaries = []
all_test_daily = []
all_test_yearly = []
all_external_daily = []
all_external_yearly = []

for fold_id in selected_folds:
    outputs = run_fold(fold_id)
    summary, test_daily, test_yearly, external_daily, external_yearly = outputs
    fold_summaries.append(summary)
    all_test_daily.append(test_daily)
    all_test_yearly.append(test_yearly)
    all_external_daily.append(external_daily)
    all_external_yearly.append(external_yearly)

run_dir = Path(args.run_dir)
summary_df = pd.DataFrame(fold_summaries).sort_values("fold")
summary_df.to_csv(run_dir / "selected_folds_summary.csv", index=False)
pd.concat(all_test_daily, ignore_index=True).to_csv(
    run_dir / "selected_folds_test_daily_metrics.csv",
    index=False,
)
pd.concat(all_test_yearly, ignore_index=True).to_csv(
    run_dir / "selected_folds_test_yearly_metrics.csv",
    index=False,
)
pd.concat(all_external_daily, ignore_index=True).to_csv(
    run_dir / "selected_folds_external_daily_metrics.csv",
    index=False,
)
pd.concat(all_external_yearly, ignore_index=True).to_csv(
    run_dir / "selected_folds_external_yearly_metrics.csv",
    index=False,
)

numeric_summary = summary_df.select_dtypes(include=[np.number]).drop(
    columns=["fold", "best_epoch"],
    errors="ignore",
)
mean_std = pd.DataFrame(
    {
        "mean": numeric_summary.mean(),
        "std": numeric_summary.std(ddof=1),
    }
)
mean_std.to_csv(run_dir / "selected_folds_mean_std.csv")

print("=" * 100)
print("SELECTED FOLDS SUMMARY")
display(summary_df)
print("MEAN ± SD")
display(mean_std)
print(f"All fold-level and aggregate outputs saved under: {run_dir}")


## Saved files per fold

Each `fold_XX` folder contains:

- `best.pt` and `last.pt`
- `training_history.csv`
- `test_2000_2020_pixel_predictions.npz`
- `external_2021_2023_pixel_predictions.npz`
- daily, yearly, and overall metric CSV files for both evaluation sets

The NPZ arrays use keys `y_true`, `y_pred`, `valid_mask`, and `dates`. The loss cap
does not alter `y_true` in these files. `high_threshold=5` is used only to split the
reported low/high MSE.
